## Data Ingestion

In [93]:
## Ingesting Data from PDF
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("C:\\Users\\somas\\OneDrive\\Desktop\\con_repos\\rag_imp\\ml_ds.pdf")
docs = loader.load()

In [94]:
print(f"Loaded {len(docs)} pages")
print(docs[36].page_content[3000:])

Loaded 162 pages



## Chunking Data

In [ ]:
# Chunking the Data
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
documents = text_splitter.split_documents(docs)

In [ ]:
documents

### Using huggingface token from .env

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["embedding"] = os.getenv("embedding")

### Intializing Embedding model

In [ ]:
# Creating Embeddings and Vector Store
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
# Initialize the model
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2") # 384 dimensions


### Loading VectorDB

In [ ]:
# Create the vector store
vectordb = Chroma.from_documents(
    documents=documents,
    collection_name="rag-chroma",
    embedding=embedding_model,
)
retriever = vectordb.as_retriever()

### Retrieving most relevant chunks from vectorDB

In [ ]:
question = "validation of practical application"
res = vectordb.similarity_search(question)
len(res)

### Loading  MODEL_ID = "google/gemma-3-1b-it" LLM from huggingface

In [ ]:
#  LLM: gemma-3-1b-it
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

MODEL_ID = "google/gemma-3-1b-it"  

# Tokenizer + model (CPU). You can set dtype=torch.float32 on pure CPU.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,   
)

In [ ]:
# Text-generation pipeline for LangChain
gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    min_new_tokens=38,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.05,
)


In [ ]:
# Wrap in a LangChain LLM interface
from langchain_community.llms import HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=gen_pipe)

#### From each doc trying to extract metadata like if present page.no. to add to end of chunk

In [ ]:
# ==== RAG Prompt & Chain ====
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    # Helpful for page-aware citations if metadata exists
    def one(d):
        pg = d.metadata.get("page", None)
        tag = f" [page {pg}]" if pg is not None else ""
        return d.page_content.strip() + tag
    return "\n\n---\n\n".join(one(d) for d in docs)

### Zero-shot Prompt

In [ ]:

# Define a plain (non-chat) prompt template to avoid role tags in output
prompt = PromptTemplate.from_template(
    """Use ONLY the context to answer the question. If the answer is not in the context, respond I don't know.

    You MUST:
- Enumerate ALL items relevant to the question that appear in the context.
- Output as a short bulleted list (one line per item).
- Do NOT add items not present in the context.

Question:
{question}

Context:
{context}
"""
)

# Keep your retriever the same; build a chain that accepts explicit context
generation_chain = prompt | llm | StrOutputParser()



## Few-shot prompt

In [ ]:

# --- Few-shot: two format exemplars, still "use only context" ---
prompt_fewshot = PromptTemplate.from_template(
    """You are given a question and some context. Use ONLY the context.
If the answer is not in the context, respond exactly: I don't know.

Follow this output style:
- Enumerate ALL relevant items that appear in the context.
- One item per bullet, concise.

### EXAMPLE 1
Question:
What metrics are mentioned?

Context:
• Precision
• Recall
• F1-score

Answer:
- Precision
- Recall
- F1-score

### EXAMPLE 2
Question:
List the datasets shown.

Context:
The following datasets are listed: MNIST, CIFAR-10, ImageNet.

Answer:
- MNIST
- CIFAR-10
- ImageNet

### TASK
Question:
{question}

Context:
{context}

Answer:"""
)
generation_chain_fewshot = prompt_fewshot | llm | StrOutputParser()


## Chain-of-thought (CoT) prompt

In [ ]:

# --- CoT: brief step-by-step, but return a clean <final>...</final> block we can parse ---
prompt_cot = PromptTemplate.from_template(
    """Use ONLY the context to answer the question.
If the answer is not in the context, respond exactly: I don't know.

First, think step by step VERY BRIEFLY using the context.
Then output the final bulleted list wrapped in <final>...</final>.

Rules:
- Enumerate ALL items relevant to the question that appear in the context.
- One bullet per line, concise.
- Do NOT add items not present in the context.

Question:
{question}

Context:
{context}

Reasoning (brief):
1) Identify items explicitly present.
2) Remove anything not in context.

<final>
-  <!-- put only the final bullets here; no extra text -->
</final>
"""
)
generation_chain_Cot = prompt | llm | StrOutputParser()




#### This is mainly extracting content between final tags during  COT since it provides reasoning and with final tag.

In [ ]:
import re
def extract_final(text: str) -> str:
    m = re.search(r"<final>(.*?)</final>", text, flags=re.DOTALL|re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return text.strip()  # fallback if tags missing

def generate_answer(chain, q: str) -> str:
    # Retrieve & format context exactly like your base flow
    docs = retriever.invoke(q)
    ctx = format_docs(docs)
    out = chain.invoke({"question": q, "context": ctx})
    return out, ctx


In [ ]:
# Helper to run once, show the exact retrieved context, and generate
def answer_with_trace(q: str):
    docs = retriever.invoke(q)         # same retriever as before
    ctx = format_docs(docs)                            # exactly what goes into the prompt
    ans = generation_chain.invoke({"question": q, "context": ctx})

    print("\nuser question:\n")
    print(q)
    print("\nretrieved context:\n")
    print(ctx if len(ctx) < 4000 else ctx[:4000] + "\n...[truncated]...")
    print("\nllm output:\n")
    print(ans)

# ==== Ask questions ====
question = "what is machine learning?"
answer_with_trace(question)

### Loading Ground Truth Dataset from Huggingface

In [ ]:
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np
df = pd.read_csv("hf://datasets/prsdm/Machine-Learning-QA-dataset/ML-101-QandA.csv")

In [ ]:

# === Load evaluation dataset ===

# Reference question/answer columns
questions = df["Question"].tolist()
gold_answers = df["Answer"].tolist()

# Embedding model for semantic comparison
eval_embedder = SentenceTransformer("all-MiniLM-L6-v2")

### Evaluation metrics

In [ ]:
import numpy as np
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# === Load evaluation dataset (only first 10 rows) ===
sample_df = df.head(10)  #
questions = sample_df["Question"].tolist()
gold_answers = sample_df["Answer"].tolist()

# Embedding model for semantic comparison
eval_embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Storage for metric results
results = []

for q, gold in tqdm(zip(questions, gold_answers), total=len(questions), desc="Evaluating RAG (10 samples)"):
    # --- Retrieve context ---
    retrieved_docs = retriever.invoke(q)
    retrieved_texts = [d.page_content for d in retrieved_docs]
    context = "\n\n".join(retrieved_texts)
    
    # --- Generate answer ---
    gen_answer = generation_chain.invoke({"question": q, "context": context})

    # --- Compute embeddings ---
    q_emb = eval_embedder.encode([q])
    gold_emb = eval_embedder.encode([gold])
    gen_emb = eval_embedder.encode([gen_answer])
    ctx_embs = eval_embedder.encode(retrieved_texts)

    # --- Retrieval metrics ---
    ctx_rel = float(np.mean(cosine_similarity(q_emb, ctx_embs)))   # Context Relevance
    ctx_rec = float(np.max(cosine_similarity(gold_emb, ctx_embs))) # Context Recall

    # --- Generation metrics ---
    faith = float(np.mean(cosine_similarity(gen_emb, ctx_embs)))   # Faithfulness (Groundedness)
    ans_rel = float(cosine_similarity(gen_emb, gold_emb)[0][0])    # Answer Relevance

    # --- End-to-end metric ---
    ans_corr = (2 * faith * ans_rel) / (faith + ans_rel + 1e-9)    # Answer Correctness

    results.append({
        "Question": q,
        "Context Relevance": round(ctx_rel, 3),
        "Context Recall": round(ctx_rec, 3),
        "Faithfulness": round(faith, 3),
        "Answer Relevance": round(ans_rel, 3),
        "Answer Correctness": round(ans_corr, 3)
    })

# === Average metrics ===
import pandas as pd

results_df = pd.DataFrame(results)
avg_metrics = results_df.drop(columns=["Question"]).mean().round(3)

print("\n=== Average RAG Evaluation Metrics (3 samples) ===\n")
print("\n COT Metrics")
print(avg_metrics.to_frame().T)
